<a href="https://colab.research.google.com/github/xidoudou/ai-agents-in-langgraph/blob/main/Lesson_4_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4: Persistence and Streaming

In [9]:
from dotenv import load_dotenv

_ = load_dotenv()

In [10]:
import os
from google.colab import userdata
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [11]:
!pip install langchain_openai

In [12]:
!pip install langchain langchain-community langgraph tavily-python

In [13]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

/tmp/ipykernel_4916/2233616850.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [14]:
tool = TavilySearchResults(max_results=2)

/tmp/ipykernel_4916/4289725543.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=2)


In [15]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [16]:
!pip install langgraph.checkpoint.sqlite

In [17]:
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string(":memory:")

In [18]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [19]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
model = ChatOpenAI(model = "gpt-6-astra")


In [27]:
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [28]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [29]:
thread = {"configurable": {"thread_id": "1"}}

In [30]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content=[{'arguments': '{"query":"current weather San Francisco California temperature today"}', 'call_id': 'call_M0kJnxrpUj2kkqQcrvtEgsRz', 'name': 'tavily_search_results_json', 'type': 'function_call', 'id': 'fc_08a3511e0eb41331006ab251b1dc4887d0b36a33d49cd5f83c', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_08a3511e0eb41331006ab251aeef1087d0ba9fc10c22192a7d', 'created_at': 1790071214.0, 'metadata': {}, 'model': 'gpt-6-astra', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-astra'}, id='resp_08a3511e0eb41331006ab251aeef1087d0ba9fc10c22192a7d', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather San Francisco California temperature today'}, 'id': 'call_M0kJnxrpUj2kkqQcrvtEgsRz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 154, 'output_tokens': 28, 'total_tokens': 182, 'input_token_details': {'cache_crea

In [31]:

messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'arguments': '{"query":"Los Angeles CA current weather conditions site:forecast.weather.gov/MapClick.php"}', 'call_id': 'call_WF1PvTEwPzFSg52hT1Py2kX4', 'name': 'tavily_search_results_json', 'type': 'function_call', 'id': 'fc_08a3511e0eb41331006ab251d3080c87d08111e3146a079d50', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_08a3511e0eb41331006ab251cff8c487d0b7aaaa80a49ba152', 'created_at': 1790071248.0, 'metadata': {}, 'model': 'gpt-6-astra', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-astra'}, id='resp_08a3511e0eb41331006ab251cff8c487d0b7aaaa80a49ba152', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'Los Angeles CA current weather conditions site:forecast.weather.gov/MapClick.php'}, 'id': 'call_WF1PvTEwPzFSg52hT1Py2kX4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1719, 'output_tokens': 36

In [33]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'type': 'text', 'text': '**Los Angeles** looks warmer based on the weather information we found.', 'annotations': [], 'id': 'msg_08a3511e0eb41331006ab251f81a3487d091aca0ba77242518', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_08a3511e0eb41331006ab251f3a2f887d08ec9b985fc84d3e0', 'created_at': 1790071283.0, 'metadata': {}, 'model': 'gpt-6-astra', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-astra'}, id='resp_08a3511e0eb41331006ab251f3a2f887d08ec9b985fc84d3e0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4495, 'output_tokens': 18, 'total_tokens': 4513, 'input_token_details': {'cache_creation': 61, 'cache_read': 4431}, 'output_token_details': {'reasoning': 0}})]}


In [34]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'type': 'text', 'text': 'Which two things are you comparing?', 'annotations': [], 'id': 'msg_03bdff4575e127a5006ab251fcf8a487d0a1baa31d4d6c6d19', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_03bdff4575e127a5006ab251fb8ea487d09f956dcee3b8a675', 'created_at': 1790071291.0, 'metadata': {}, 'model': 'gpt-6-astra', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-astra'}, id='resp_03bdff4575e127a5006ab251fb8ea487d09f956dcee3b8a675', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 11, 'total_tokens': 163, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})]}


## Streaming tokens

In [1]:
!pip install langgraph-checkpoint-sqlite

In [5]:
!pip install -U langgraph-checkpoint-sqlite
import langgraph.checkpoint
print(dir(langgraph.checkpoint))

['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [22]:
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

async with AsyncSqliteSaver.from_conn_string(":memory:") as memory:
    abot = Agent(model, [tool], system=prompt, checkpointer=memory)

    messages = [HumanMessage(content="What is the weather in sf?")]
    thread = {"configurable": {"thread_id": "4"}}

    async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
        print(event)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3551: LangChainDeprecationWarning: astream_events version='v1' is deprecated. Use version='v2' or astream instead.
  await eval(code_obj, self.user_global_ns, self.user_ns)


{'event': 'on_chain_start', 'run_id': '01a0c895-fb46-7d42-85b5-9fbc2c3bc8d7', 'name': 'LangGraph', 'tags': [], 'metadata': {}, 'data': {'input': {'messages': [HumanMessage(content='What is the weather in sf?', additional_kwargs={}, response_metadata={})]}}, 'parent_ids': []}
{'event': 'on_chain_start', 'name': 'llm', 'run_id': '01a0c895-fb4d-7760-ad71-7814320fd7e9', 'tags': ['graph:step:1'], 'metadata': {'thread_id': '4', 'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'llm', 'langgraph_triggers': ('branch:to:llm',), 'langgraph_path': ('__pregel_pull', 'llm'), 'langgraph_checkpoint_ns': 'llm:09a0854f-4fa6-433c-8086-253f15c5674c'}, 'data': {'input': {'messages': [HumanMessage(content='What is the weather in sf?', additional_kwargs={}, response_metadata={})]}}, 'parent_ids': []}
{'event': 'on_chat_model_start', 'name': 'ChatOpenAI', 'run_id': '01a0c895-fb51-7db2-88ab-8aaba83ad71a', 'tags': ['seq:step:1'], 'metadata': {'thread_id': '4', 'ls_integration': 'langchain_c

In [24]:
async with AsyncSqliteSaver.from_conn_string(":memory:") as memory:
    abot = Agent(model, [tool], system=prompt, checkpointer=memory)

    messages = [HumanMessage(content="What is the weather in SF?")]
    thread = {"configurable": {"thread_id": "4"}}
    async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
        kind = event["event"]
        if kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                print(content, end="|")

[{'type': 'function_call', 'name': 'tavily_search_results_json', 'arguments': '', 'call_id': 'call_IU5POP2LlwHNqtqj6AtEMFRr', 'id': 'fc_04f2c2a4a07352fa006ab253bd8c1487d097ad5023b26b8e7a', 'index': 0}]|[{'type': 'function_call', 'arguments': '{"', 'index': 0}]|[{'type': 'function_call', 'arguments': 'query', 'index': 0}]|[{'type': 'function_call', 'arguments': '":"', 'index': 0}]|[{'type': 'function_call', 'arguments': 'San', 'index': 0}]|[{'type': 'function_call', 'arguments': ' Francisco', 'index': 0}]|[{'type': 'function_call', 'arguments': ' weather', 'index': 0}]|[{'type': 'function_call', 'arguments': ' current', 'index': 0}]|[{'type': 'function_call', 'arguments': ' today', 'index': 0}]|[{'type': 'function_call', 'arguments': '"}', 'index': 0}]|Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'San Francisco weather current today'}, 'id': 'call_IU5POP2LlwHNqtqj6AtEMFRr', 'type': 'tool_call'}
Back to the model!
[{'type': 'function_call', 'name': 'tavily_search_res